# Separate YOLO + MediaPipe alternative

Keep your existing models. Upload the SAME sorted BabyTraining.zip to compare a matching MediaPipe baseline with a new YOLO-assisted alternative. Each alternative has **one final posture classifier** and its own report.

MediaPipe is tried first, then YOLO body crops, then (if needed) YOLO 2D body keypoints plus actual MediaPipe dense face landmarks. There are still **12 named inputs**, but four visibility entries use YOLO confidence on fallback frames, and head_lift_ratio stays missing without MediaPipe 3D pose. This changed feature contract requires retraining; it is not a drop-in old-model input. Generic bundled YOLO weights are not infant-fine-tuned. [Read the exact definitions and current-session reuse instructions](https://github.com/kg905/SeniorDesignProject/blob/main/docs/HYBRID_ALTERNATIVE.md).

No 95% claim is made. Use a CPU runtime, run cells in order, and download results before resetting. Existing pilot data remains exploratory.


## 1. Get the code and original grouping reference
A sparse checkout avoids downloading all 5,814 source images again. Existing checkouts are fast-forwarded without deleting local data.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
GIT_ENV = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/kg905/SeniorDesignProject.git', str(ROOT)], env=GIT_ENV, check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', 'src', 'training', 'deploy/single-model',
                    'docs', 'models', 'artifacts/dataset_audit_v2'], cwd=ROOT, env=GIT_ENV, check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=ROOT, env=GIT_ENV, check=True)
os.chdir(ROOT)
import importlib
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
print('Source ready:', ROOT)


## 2. Install dependencies

This affects this temporary Colab runtime, not your Pi. Keep only one pip cv2 distribution. If Colab requests a restart, restart after installation, rerun section 1, then continue to section 3. Do not skip an unsuccessful install.

The hybrid adds CPU Torch/torchvision and Ultralytics. Ultralytics is installed with --no-deps so it does not add a second OpenCV distribution. If a restart is required, do it before uploading data.


In [ ]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install -r /content/SeniorDesignProject/training/requirements.txt
%pip install --no-deps -e /content/SeniorDesignProject
!apt-get update -qq
!apt-get install -y -qq git-lfs libgles2 libegl1 libgl1 libportaudio2

%pip install --index-url https://download.pytorch.org/whl/cpu torch==2.9.1 torchvision==0.24.1
%pip install -r /content/SeniorDesignProject/training/requirements-hybrid.txt
%pip install --no-deps ultralytics==8.3.200


## 3. Upload your sorted ZIP to LOCAL Colab storage

Upload BabyTraining.zip, not the original unsorted Roboflow ZIP. Upload, extraction and training stay under **/content/cribwatch-runs**, not the Google Drive mount. This prevents a disconnected Drive from interrupting active file writes. **Local files disappear on a runtime reset: download your model/results in section 8 before closing/resetting Colab.**

If the ZIP is already in Drive, set ZIP_IN_DRIVE below; only that option mounts Drive and copies the source ZIP locally. Otherwise leave it empty and select the ZIP from your computer. Existing Drive reviews/previous runs are not overwritten.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
os.chdir(ROOT)
import importlib
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
from google.colab import files
from cribwatch.colab_io import local_zip_run
ZIP_IN_DRIVE = ''  # optional: '/content/drive/MyDrive/BabyTraining.zip'
if ZIP_IN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
RUN, ZIP_PATH = local_zip_run(source_zip=ZIP_IN_DRIVE or None)
print('Verified local ZIP:', ZIP_PATH)
print('LOCAL run:', RUN)
print('Download section 8 outputs before resetting/disconnecting this runtime.')


## 4. Import folders as labels and check counts

The importer accepts one enclosing BabyTraining folder (or direct class folders). It verifies images, preserves your labels, keeps unknowns out of posture training, and quarantines conflicting copies. Hash matches to the original audit preserve known duplicate/source groups; new images get conservative duplicate grouping. None of this invents baby IDs.


In [ ]:
from cribwatch.assets import safe_unzip
from cribwatch.folder_data import import_folders
from cribwatch.data import read_csv
EXTRACTED = RUN / 'sorted-images'
safe_unzip(ZIP_PATH, EXTRACTED)
AUDIT = RUN / 'folder-audit'
summary = import_folders(EXTRACTED, AUDIT, reference_index=ROOT / 'artifacts/dataset_audit_v2/images.csv')
print('Your folder counts:', summary['folder_counts'])
print('Eligible exact-unique posture images:', summary['eligible_unique_counts'])
print('Matched to original audit:', summary['matched_original_images'])
print('Conservative source groups:', summary['source_groups'])
print('Conflicting images excluded:', summary['conflicting_images'])
print('Ignored files:', summary['ignored_files'][:10])
if summary['conflicting_images']:
    print('Inspect', AUDIT / 'conflicts.json', 'and correct the computer folders for a future run.')
if any(summary['eligible_unique_counts'].get(label, 0) == 0 for label in ['back','side','prone']):
    raise ValueError('Need non-conflicting images in ALL THREE posture folders before training.')
MANIFEST = AUDIT / 'images.csv'
print('Labels saved:', MANIFEST)


## 5. Train the baseline and separate hybrid alternative

The same labels/groups/split seed are used. Each classifier recipe is selected on validation only. Reports compare two preplanned alternatives; test scores do not select recipes. Reusing the same paths resumes completed features/models.

In [ ]:
from cribwatch.hybrid_experiment import run_hybrid_comparison
BASELINE_FEATURES = globals().get('BASELINE_FEATURES', globals().get('FEATURES'))
if BASELINE_FEATURES is not None and not Path(BASELINE_FEATURES).exists():
    BASELINE_FEATURES = None
HYBRID_RUN = RUN / 'hybrid-alternative-v1'
YOLO_WEIGHTS = ''  # optional: trusted compatible infant-tuned COCO-17 pose checkpoint
hybrid = run_hybrid_comparison(
    MANIFEST, HYBRID_RUN, ROOT / 'models', existing_features=BASELINE_FEATURES,
    pose_variant='lite', yolo_weights=YOLO_WEIGHTS or None,
)
MODEL = Path(hybrid['model_dir'])
FEATURES = Path(hybrid['features'])
report = hybrid['report']
print('Hybrid classifier:', MODEL)
print('Matching baseline:', hybrid['baseline_model_dir'])
print('Comparison JSON:', hybrid['comparison_path'])


## 6. Compare the measured results

More usable landmarks do not automatically mean better classification. Check prone recall, rejected frames and timing. Share hybrid_comparison.json for diagnosis.

In [ ]:
for name, info in hybrid['comparison']['variants'].items():
    r, d = info['test'], info['diagnostics']
    print('\n', name.upper())
    print('Pose coverage across eligible images:', f"{d['pose_coverage']:.1%}")
    print('Test accuracy (unknown = error):', f"{r['accuracy_all_images_unknown_counts_as_error']:.1%}")
    print('Test prediction coverage:', f"{r['coverage']:.1%}")
    for label, values in r['per_class'].items():
        print(label, 'recall:', f"{values['recall']:.1%}")
    print('Rejection reasons:', r['rejection_reasons'])
    print('Usable crop recoveries:', d['usable_yolo_crop_recoveries'])
    print('Usable YOLO keypoint recoveries:', d['usable_yolo_keypoints_recoveries'])
    print('Median inference milliseconds on THIS machine:', d['inference_median_ms_this_machine'])
print('Recommendation uses validation only:', hybrid['comparison']['recommended_by_validation_only'])
print('Neither pilot result establishes accuracy on unseen infants or live Pi performance.')


## 7. See example scans with a pose box and risk level

These are examples after evaluation, not a new accuracy test. Body position maps to lower/moderate/high posture risk; unknown stays unknown. The visible-landmark box is not a separate baby/airway detector. Confidence is not a medical-risk percentage.


In [ ]:
examples = [r['image_path'] for r in read_csv(MODEL / 'test_predictions.csv')][:6]
ANNOTATED = RUN / 'annotated-scans'
subprocess.run([sys.executable, '-m', 'cribwatch.showcase', 'image', '--model', str(MODEL / 'posture.joblib'),
                '--models', str(ROOT / 'models'), '--showcase', '--output', str(ANNOTATED), *examples], check=True)
from IPython.display import display
from PIL import Image
for image in sorted(ANNOTATED.glob('*.jpg')):
    display(Image.open(image))


## 8. DOWNLOAD the trained/evaluated Pi package before ending the runtime

This packages one classifier, MediaPipe assets, report, scan/receiver scripts and instructions. Local results do not survive a runtime reset. Download the ZIP and keep it on your computer. Optional backup to Drive runs only if BACKUP_TO_DRIVE is enabled below, after local processing is complete; a Drive backup failure does not remove your local outputs.

Physical Pi camera/GPIO/Wi-Fi and continuous-event tests remain necessary. No 95% or V10 artifact is fabricated.

This separate ZIP bundles one hybrid classifier, its MediaPipe assets and YOLO weights, extra requirements, license attribution and HYBRID.md. Install the extra dependencies from HYBRID.md on the Pi before running. Old model files remain unchanged.


In [ ]:
ZIP_OUT = RUN / 'SIDS_Pi5_Hybrid_Alternative.zip'
subprocess.run([sys.executable, str(ROOT / 'deploy/single-model/build_handoff.py'),
                '--model', str(MODEL), '--output', str(ZIP_OUT)], check=True)
files.download(str(ZIP_OUT))
print('Local download source:', ZIP_OUT)
print('Keep the downloaded ZIP on your computer; /content storage is temporary.')
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    backup = Path('/content/drive/MyDrive/CribWatchManual') / RUN.name
    backup.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_OUT, backup / ZIP_OUT.name)
    print('Copied to Drive:', backup / ZIP_OUT.name)


## Adding more images later

Add photos to the same class folders on your computer, create a ZIP of the entire updated BabyTraining folder, and start a new run. The new run imports your current folder choices and leaves old local runs intact while the runtime exists; download results before resetting; adding files does not automatically retrain a deployed model. The original review-widget CSV is not needed for this workflow.

After installation on the Pi desktop, the package supports:

```bash
python monitor_pi5.py --model models --models models --picamera --display --showcase --alarm-risk moderate
```

Adopt the bundled TCP 5000 Pi 4 receiver and shared-key contract before adding `--alarm-host <Pi4-IP>`. Verify actual sound, risk timing, latching/mute/acknowledgement and camera/network faults. The attached PDF's V10 notebook/receiver source were not supplied; compatibility with unseen code is not claimed. Live screenshots demonstrate operation; only independent evaluation can support a 95% posture claim. No SIDS-prevention claim is made.
